# Evaluating Response Relevancy with Ragas

In advanced Retrieval-Augmented Generation (RAG) systems, the quality of the final answer is often more critical than the retrieval step itself. While a system might successfully retrieve highly relevant source documents, if the Large Language Model (LLM) fails to synthesize that information concisely or drifts into tangential details, the user experience suffers. This notebook introduces **Answer Relevancy**, a crucial metric for evaluating whether an LLM's generated response directly and completely addresses the user's original query without unnecessary filler or deviation.

This concept is paramount in building robust RAG pipelines. A low relevancy score indicates that even if the underlying source documents were perfect, the generation step failed to focus. For advanced architectures like LangGraph, incorporating metrics such as Answer Relevancy allows developers to build sophisticated evaluation loops. You can implement a "relevance gate" where the graph checks the generated answer against predefined thresholds; if the relevancy is too low, the system can trigger a self-correction loop (e.g., re-prompting the LLM with stricter instructions) or flag the output for human review, significantly improving reliability and user trust.

By mastering metrics like Answer Relevancy using frameworks such as `ragas`, you move beyond simple accuracy checks. You learn to evaluate the *quality of communication*—ensuring that the AI is not just knowledgeable, but also precise, focused, and actionable. This skill set is essential for deploying production-grade RAG applications that meet high standards of professional utility.

### Learning Objectives
Upon completing this notebook, you will be able to:
*   Understand the concept of Answer Relevancy in the context of LLM evaluation.
*   Utilize the `ragas` library to instantiate and run advanced metrics like `AnswerRelevancy`.
*   Analyze different response patterns (e.g., perfect answers vs. tangential filler) and interpret their corresponding relevancy scores.
*   Identify how incorporating relevance checks can improve the robustness of complex RAG pipelines or LangGraph agents.


### Setup for Answer Relevancy Scoring

This cell initializes the necessary components—the OpenAI client, LLM, and embedding model—required to calculate answer relevancy. It then instantiates `AnswerRelevancy`, which is a metric class from Ragas designed to evaluate how well generated answers address the original question.


In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy

# Initialize the asynchronous OpenAI client for API calls
client = AsyncOpenAI()
# Factory function to get an LLM instance (using a placeholder model name)
llm = llm_factory("gpt-5-mini", client=client)
# Factory function to get an embedding model instance
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

# Instantiate the AnswerRelevancy metric scorer using the initialized LLM and embeddings
scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)


d:\rag-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Code Explanation

This cell demonstrates the evaluation of response relevancy using an asynchronous scoring mechanism (`await scorer.ascore`). It tests a scenario where the generated answer correctly addresses the core question but includes significant tangential information (e.g., discussing Sydney/Melbourne compromise), which is expected to lower the calculated relevance score.


In [ ]:
# Example 1 : Response answers the question but drifts into tangential information
# Mentioning the compromise between Sydney and Melbourne dilutes relevancy
result = await scorer.ascore(
    user_input="What is the capital of Australia?",
    response="Australia is a large country in the Southern Hemisphere. It has many major cities including Sydney, Melbourne, and Brisbane. Canberra is the capital city, chosen as a compromise between Sydney and Melbourne. Australia also has a diverse economy driven by mining and agriculture."
)
print(f"Response Relevancy Score: {result.value}")


Response Relevancy Score: 0.9621900768990267


This cell demonstrates how to calculate the 'Response Relevancy' score using an asynchronous scoring mechanism (`scorer.ascore`). It tests a scenario where the provided response is highly relevant and directly answers the user's question, allowing us to quantify the quality of the generated answer.


In [ ]:
# Example 2: Response is direct and precisely answers the question with no filler
result = await scorer.ascore(
    user_input="When was the first Super Bowl played?",
    response="The first Super Bowl was played on January 15, 1967."
)
print(f"Response Relevancy Score: {result.value}")


Response Relevancy Score: 1.0000000000000002


### Code Explanation

This cell demonstrates the use of a response relevancy scorer to evaluate how well a generated answer addresses the user's specific query. It calculates a score based on whether the provided `response` actually contains information relevant to the topic requested in `user_input`, even if the response discusses related general facts.


In [ ]:
# Example 3 : Response talks about water as a topic but never states its boiling point
result = await scorer.ascore(
    user_input="What is the boiling point of water?",
    response="Water is a fascinating substance found all over the Earth. It is essential for all known forms of life and covers about 71 percent of the Earth's surface. Water is found in oceans, rivers, lakes, and glaciers and plays a key role in regulating climate."
)
# Print the calculated relevancy score value
print(f"Response Relevancy Score: {result.value}")


Response Relevancy Score: 0.26754335611869756
